# Aula 16 - Problemas Eulerianos e Inspeção de Infraestrutura ATEX

Mapeamento de um circuito de anel da malha da Estação e cálculo do voo ótimo de inspeção do drone de detecção de vazamentos.

In [ ]:
from collections import defaultdict
from typing import List, Tuple

class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)
        
    def adicionar_duto(self, u: str, v: str, id_duto: str):
        # Como a inspeção é via espaço aéreo/robô percorrendo a tubulação,
        # modelamos o trajeto como não-dirigido
        self.adj[u].append((v, id_duto))
        self.adj[v].append((u, id_duto))
        
    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito

# Modelagem de Sub-Malha fechada (Anéis topológicos) para garantir graus pares
g_insp = GrafoInspecaoEuleriano()

# Anel Periférico (Rota Externa)
g_insp.adicionar_duto("Base", "C-101", "Duto_01")
g_insp.adicionar_duto("C-101", "TK-101", "Duto_02")
g_insp.adicionar_duto("TK-101", "MAN-101", "Duto_03")
g_insp.adicionar_duto("MAN-101", "D-101", "Duto_04")
g_insp.adicionar_duto("D-101", "Base", "Duto_05")

# Anel Interno de Alta Pressão (Conectado nos nós C-101 e MAN-101)
g_insp.adicionar_duto("C-101", "C-102", "Duto_06")
g_insp.adicionar_duto("C-102", "TK-102", "Duto_07")
g_insp.adicionar_duto("TK-102", "MAN-101", "Duto_08")
g_insp.adicionar_duto("MAN-101", "C-101", "Duto_09") # Fechamento interno

eul, imp = g_insp.verificar_euleriano()
print(f"Grafo de H2 é Euleriano: {eul} | Vértices Ímpares: {imp}")

rota_drone = g_insp.calcular_circuito_hierholzer("Base")
print("Circuito Euleriano de Inspeção ATEX:\n", " -> ".join(rota_drone))
